# B 연습 노트북 — `poly_tomsg` CPA 단계별 실습

원본 `mlkem-cpa-poly_tomsg.ipynb`는 그대로 두고, 이 파일을 직접 입력하는 실습지로 써. 각 TODO를 한 셀씩 채우고 실행해. 원본은 답안과 PQClean 산술 참고용이야.

## 학습 목표
- ciphertext와 비밀 NTT 벡터로 `poly_tomsg` 입력 `mp`를 재현한다.
- 비밀계수 후보마다 `mp`와 HW 누출 모델을 만든다.
- Pearson CPA, 후보 점수, rank를 계산한다.
- 단일계수 oracle 실험과 blind 전체키 공격의 차이를 설명한다.

B는 `mp`에 여러 비밀계수가 함께 영향을 주기 때문에, CPA 계산 전에 **후보 중간값을 어떻게 만들었는지**가 핵심이야.


## 0. 저장 트레이스 불러오기

기존 B 일반 캡처에서 첫 100행을 사용해. 이 파일의 `traces[t]`와 `ciphertexts[t]`가 같은 요청이야. TVLA 파일이 아니라 `traces`, `ct`, `dk`가 저장된 CPA 입력이야.

`dk` 값 자체는 출력하지 않아. 아래 실습은 정답 계수를 rank로 채점하는 known-key 조건이 포함돼.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

with np.load("../traces/mlkem768-clean-O3-poly_tomsg-20261030-053826.npz") as data:
    traces_full = data["traces"].astype(np.float32)
    ciphertexts_full = data["ct"].astype(np.uint8)
    secret_key = data["dk"].tobytes()

trace_count = min(100, len(traces_full))
traces = traces_full[:trace_count]
ciphertexts = ciphertexts_full[:trace_count]
print("traces:", traces.shape, "ciphertexts:", ciphertexts.shape)


### 체크포인트 0

기대 shape은 `traces=(100, 20000)`, `ciphertexts=(100,1088)`야. `K=3`, `N=256`, `q=3329`를 기억해. 실제로 CPA에 넣는 각 행은 `(파형, 암호문)`의 쌍이야.


In [ ]:
Q, N, K = 3329, 256, 3
SECRET_VEC, SECRET_COEFF = 0, 15
OUTPUT_COEFF = 202
SAMPLE_WINDOW = (5630, 5661)


## 1. `mp` 재현용 PQClean 연산 — 직접 작성

먼저 원본의 scalar helper를 참고해 `int16` wrap, Montgomery/Barrett reduction, NTT와 압축 해제를 작성해. 이후 벡터형 helper에서는 동일 연산을 여러 ciphertext 행에 적용해.

**입력 분해:** 앞 960바이트=`u₀‖u₁‖u₂` (각 320바이트), 뒤 128바이트=`v`.

**목표:** `decrypt_mp_with_secret(ciphertexts, secret)`가 `(trace_count,256)` 배열을 반환하게 해. 이 값이 다음에 공격할 `poly_tomsg` 입력이야. 수식의 mod-q 결과뿐 아니라 C의 `int16` 표현도 맞춰야 해.


In [ ]:
# TODO: 원본 A/B 노트북을 참고해 exact-arithmetic helper를 직접 입력해.
# scalar group: to_i16, montgomery_reduce, fqmul, barrett_reduce, ntt
# decode group: poly_frombytes, polyvec_decompress_vec, shat_from_dk
# vector group: wrap16, fqmul_vec, barrett_vec, ntt_vec,
#               basemul_vec, invntt_vec, decrypt_mp_with_secret
# 한 번에 다 쓰기보다 그룹별로 입력하고, 다음 체크포인트를 실행해.


In [ ]:
# 체크포인트 1: 정답 키로 mp를 재현하는 연산이 동작하는지 확인해.
secret_coefficients = np.asarray(shat_from_dk(secret_key), dtype=np.int64)
mp_true = decrypt_mp_with_secret(ciphertexts, secret_coefficients)
assert mp_true.shape == (trace_count, N)
print("mp shape:", mp_true.shape)


## 2. 타깃 계수의 영향량 만들기 — 직접 작성

기준 키의 타깃 계수만 `0`으로 둔 복호화 결과를 `mp_zero`, `1`로 둔 결과를 `mp_unit`이라 하자. 두 결과 차이를 `q`로 나눈 잔여값으로 두면 타깃 후보 `g`에 대해

`mp_g = (mp_zero + g * delta) mod q`

로 중간값을 생성할 수 있어.

**여기서 oracle 문제가 생겨:** 기준 키의 다른 767개 계수는 그대로 남아. 이 실험은 그 값들을 이미 아는 조건에서 타깃 한 계수를 비교하는 거야. 식을 이해한 뒤 코드를 직접 채워.


In [ ]:
def build_secret_sensitivity(ciphertexts, secret_reference, secret_vector, secret_coefficient):
    # TODO: 타깃 계수만 0/1로 설정한 두 mp를 구하고 delta를 반환해.
    raise NotImplementedError


def build_candidate_mp(mp_zero, delta, guesses, output_coefficient):
    # TODO: 선택한 출력 계수에서 각 후보의 centered mp를 계산해.
    raise NotImplementedError


In [ ]:
# 체크포인트 2: 후보 중간값이 원본 계산과 일치하는지 확인해.
true_secret_coeff = int(secret_coefficients[SECRET_VEC, SECRET_COEFF])
mp_zero, delta = build_secret_sensitivity(ciphertexts, secret_coefficients, SECRET_VEC, SECRET_COEFF)
true_candidate_mp = build_candidate_mp(mp_zero, delta, [true_secret_coeff], OUTPUT_COEFF)
assert true_candidate_mp.shape == (1, trace_count)
assert np.array_equal(true_candidate_mp[0], mp_true[:, OUTPUT_COEFF])
print("candidate mp reconstruction OK")


## 3. `poly_tomsg` 입력의 leakage model — 직접 작성

후보별 centered `mp`를 C처럼 `uint32_t` 비트 패턴으로 해석하고, 네 바이트 각각의 Hamming weight를 더해 `model[g,t]`를 만들어. 이건 입력 워드의 HW 누출이라는 가설이야.

`OUTPUT_COEFF=202`는 **관측하는 mp 위치**, `SECRET_COEFF=15`는 **추정하려는 키 위치**야. 혼동하지 않도록 shape을 확인해.


In [ ]:
def build_tomsg_input_models(candidate_mp):
    # TODO: uint32 비트 패턴 -> 4개 byte -> byte-HW 합
    raise NotImplementedError


def calculate_pearson_correlations(candidate_models, measured_traces):
    # TODO: 각 후보 × 각 샘플의 Pearson 상관을 계산해.
    raise NotImplementedError


In [ ]:
# 체크포인트 3: 한 후보 모델과 파형의 sample-by-sample 상관.
one_model = build_tomsg_input_models(true_candidate_mp)
one_correlation = calculate_pearson_correlations(one_model, traces[:, SAMPLE_WINDOW[0]:SAMPLE_WINDOW[1]])
assert one_model.shape == (1, trace_count)
assert one_correlation.shape == (1, SAMPLE_WINDOW[1] - SAMPLE_WINDOW[0])
print("model/correlation shapes:", one_model.shape, one_correlation.shape)


## 4. 한 좌표 CPA 실행 — 직접 작성

이번엔 전체 sweep 전에 한 비밀계수만 공격해.

1. 앞 체크포인트에서 계산한 `secret_coefficients`, `mp_true`, `mp_zero`, `delta`를 재사용한다.
2. `mp_zero, delta`와 3329개 후보의 `candidate_mp`를 만든다.
3. 후보별 HW 모델을 만든다.
4. `SAMPLE_WINDOW`의 파형과 상관을 계산하고 후보별 최대 절댓값으로 점수화한다.
5. 정답 계수의 rank, 최고 후보, peak sample을 출력한다.

정답 키로 `mp_true`/rank를 확인할 수 있지만, 그 값은 후보 점수 모델의 다른 767계수에도 들어가 있어. 그러니 rank 0이어도 oracle 조건이라고 설명해야 해.


In [ ]:
# TODO: 0..3328 후보의 candidate_mp, HW model, correlation, score를 계산해.
# 앞 체크포인트의 secret_coefficients/mp_zero/delta를 재사용해.
# 후보별 최고 |corr| 점수로 best_guess와 true_rank, peak sample을 계산해.
# 기대: model shape=(Q, trace_count), correlation shape=(Q, window 길이)


### 체크포인트 4

기존 100-trace 캡처의 저장 결과에서는 이 좌표가 rank 0이고 true-candidate의 최고점은 샘플 5644 부근이야. 네 결과가 다르면 먼저 `mp` 재현 assertion, trace/ciphertext 행 정렬, centered `mp`의 `uint32` 변환, 그리고 window 인덱스를 확인해. 키 값은 출력하지 않아도 돼.


## 5. 더 많은 계수와 blind 한계 — 선택 실습

원본의 전체 스윕을 읽고 `ORACLE_BASELINE=True`에서 한 좌표의 reference secret, 후보, POI가 어떻게 정해지는지 표시해. 그다음 `False`의 좌표하강을 소수 좌표만으로 시험해 봐.

**정답으로 고정한 다른 계수가 하나라도 있는지, POI를 실제 타깃 값으로 골랐는지, 공격 단계에서 정답을 출력/참조했는지** 체크리스트를 만들어. blind 복구를 평가할 때는 공격용 키를 후보 점수·POI·부호에 넣지 말고, 공격이 끝난 뒤에만 사후 채점해야 해.

B의 추가 난점은 `mp`가 비밀계수 여러 개에 의존한다는 거야. 다른 767개를 0으로 놓은 한 계수 공격이 실패해도, 그것만으로 모든 blind 전략이 불가능하다는 뜻은 아니지만, profiling POI만 옮겼다고 의존성 문제가 해결되는 것도 아니야.

완성 답안과 oracle full-sweep 예시는 기존 `mlkem-cpa-poly_tomsg.ipynb`에 남아 있어.
